# Belt Sentinel — Train on ColabTrain the conveyor belt damage detector on a free Colab GPU, then download theweights into `backend/models/`.**Why here rather than locally:** `yolo11s` is the better model, but on an8 GB Apple silicon laptop it needs ~17 hours for 60 epochs because it pages todisk. A free Colab T4 does 120 epochs in roughly 1–2 hours.**Before you start:** Runtime → Change runtime type → **T4 GPU**.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader!pip install -q ultralytics roboflow

## 2. Get the dataPaste your Roboflow API key below (Settings → API keys). The free Public planis enough — these are public CC BY 4.0 datasets.

In [ ]:
import osos.environ["ROBOFLOW_API_KEY"] = ""  # <-- paste your keyassert os.environ["ROBOFLOW_API_KEY"], "Set your Roboflow API key above"

In [ ]:
# Pull the same datasets the local pipeline uses.from roboflow import Roboflowfrom pathlib import PathRAW = Path("data/raw"); RAW.mkdir(parents=True, exist_ok=True)rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])DATASETS = [    ("conveyor-belt-damage-sample", "sample-wy2mp", "conveyor-belt-damage", 1),    ("conveyor-belt-damage-test",   "test-yfiry",   "conveyor-belt-damage-ucjlj", 1),]for slug, workspace, project, version in DATASETS:    target = RAW / slug    if (target / "data.yaml").exists():        print("already have", slug); continue    try:        p = rf.workspace(workspace).project(project)        p.version(version).download("yolov8", location=str(target), overwrite=True)        print("downloaded", slug)    except Exception as e:        print("failed", slug, e)

## 3. Unify the class namesPublishers label the same defect differently ("Large Tear", "rip", "Tear"), soeverything is remapped onto one vocabulary. Classes with no examples are**excluded** — a model must not advertise a class it was never shown.

In [ ]:
%%writefile classes.pyCLASS_NAMES = ["tear", "hole", "scratch", "crack", "belt_joint", "joint_damage"]ALIASES = {    "tear": "tear", "tears": "tear", "large tear": "tear", "small tear": "tear",    "rip": "tear", "longitudinal tear": "tear", "belt tear": "tear",    "hole": "hole", "large hole": "hole", "small hole": "hole",    "puncture": "hole", "perforation": "hole", "impact damage": "hole",    "scratch": "scratch", "scratches": "scratch", "abrasion": "scratch", "wear": "scratch",    "crack": "crack", "cracks": "crack", "fissure": "crack",    "belt joint": "belt_joint", "joint": "belt_joint", "splice": "belt_joint",    "seam": "belt_joint", "fastener": "belt_joint",    "joint damage": "joint_damage", "damaged joint": "joint_damage",    "splice failure": "joint_damage",}def normalise(label):    return label.strip().lower().replace("_"," ").replace("-"," ").replace("/"," ").strip()def canonical(label):    k = normalise(label)    if k in ALIASES: return ALIASES[k]    for alias, target in ALIASES.items():        if alias in k: return target    return None

In [ ]:
import shutil, collections, random, yamlfrom pathlib import Pathfrom classes import CLASS_NAMES, canonicalRAW, OUT = Path("data/raw"), Path("data/merged")index_of = {n: i for i, n in enumerate(CLASS_NAMES)}counts = collections.Counter()pending = []rng = random.Random(1337)for ds in sorted(p for p in RAW.iterdir() if p.is_dir()):    cfg = yaml.safe_load((ds / "data.yaml").read_text())    names = cfg["names"]    if isinstance(names, dict):        names = [names[k] for k in sorted(names, key=int)]    mapping = {}    for i, raw in enumerate(names):        t = canonical(raw)        print(f"  {ds.name}: {raw!r} -> {t or 'UNMAPPED (skipped)'}")        if t: mapping[i] = index_of[t]    for split in ("train", "valid", "test"):        img_dir, lbl_dir = ds / split / "images", ds / split / "labels"        if not img_dir.is_dir(): continue        for img in sorted(img_dir.iterdir()):            lbl = lbl_dir / f"{img.stem}.txt"            if not lbl.exists(): continue            kept = []            for line in lbl.read_text().splitlines():                parts = line.split()                if len(parts) < 5: continue                src = int(float(parts[0]))                if src not in mapping: continue                kept.append(" ".join([str(mapping[src]), *parts[1:]]))                counts[CLASS_NAMES[mapping[src]]] += 1            pending.append((img, "valid" if split == "test" else split, kept))trained = [c for c in CLASS_NAMES if counts[c] > 0]reindex = {index_of[c]: i for i, c in enumerate(trained)}print("\nDistribution:")for c in CLASS_NAMES:    print(f"  {c:14} {counts[c]}")print(f"\nEmitting: {trained}")print(f"Excluded (no examples): {[c for c in CLASS_NAMES if not counts[c]]}")if OUT.exists(): shutil.rmtree(OUT)for s in ("train", "valid"):    (OUT / s / "images").mkdir(parents=True); (OUT / s / "labels").mkdir(parents=True)for i, (img, split, lines) in enumerate(pending):    stem = f"{i:06d}_{img.stem}"    shutil.copy2(img, OUT / split / "images" / f"{stem}{img.suffix}")    out = [f"{reindex[int(l.split()[0])]} {' '.join(l.split()[1:])}" for l in lines]    (OUT / split / "labels" / f"{stem}.txt").write_text("\n".join(out) + ("\n" if out else ""))(OUT / "data.yaml").write_text(yaml.safe_dump({    "path": str(OUT.resolve()), "train": "train/images", "val": "valid/images",    "nc": len(trained), "names": trained,}, sort_keys=False))print("\nWrote", OUT / "data.yaml")

## 4. Train`yolo11s` at batch 32 — a T4 has 16 GB, so there is no reason to shrink thebatch the way the 8 GB laptop has to. Augmentation is tuned for belt imagery:near-monochrome, dusty, and directional.

In [ ]:
from ultralytics import YOLOmodel = YOLO("yolo11s.pt")model.train(    data="data/merged/data.yaml",    epochs=120, imgsz=640, batch=32, device=0,    project="runs", name="belt_v1", patience=30, exist_ok=True,    hsv_h=0.010,      # belt rubber is essentially hueless    hsv_s=0.40,    hsv_v=0.60,       # wide value jitter: lighting is the big variable    degrees=3.0,      # a belt is near axis-aligned    translate=0.12, scale=0.45, shear=2.0,    fliplr=0.5, flipud=0.0,   # never flip vertically: belts run one way    mosaic=1.0, close_mosaic=15,    erasing=0.25,     # simulates occlusion by ore on the belt)

## 5. Evaluate

In [ ]:
metrics = model.val(data="data/merged/data.yaml", imgsz=640, device=0)print(f"mAP@.5      {metrics.box.map50*100:.1f}%")print(f"mAP@.5:.95  {metrics.box.map*100:.1f}%")print(f"Precision   {metrics.box.mp*100:.1f}%")print(f"Recall      {metrics.box.mr*100:.1f}%")print()print("Baseline — Guo et al., Micromachines 2022, Table 5:")print("  YOLOv5m 82.5% @ 128 FPS   ·   Faster R-CNN 86.4% @ 7.4 FPS")

## 6. Download the weightsPut the downloaded file at `backend/models/belt_v1.pt`, set `DETECTOR=yolo` in`backend/.env`, then `./scripts/stop.sh && ./scripts/start.sh`.

In [ ]:
from google.colab import filesfiles.download("runs/belt_v1/weights/best.pt")